<div style="font-size:12pt; font-weight:bold;">Modularity Vitality Tests</div>
<div style="font-size:12pt;">Jonathan H. Morgan, Ph.D.</div>
<div style="font-size:12pt;">31 October 2025</div>

<div style="font-size:10pt; font-weight:bold;">Preamble</div>

In [13]:
import os
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Sequence, Mapping,  Any
import igraph as ig
from scipy.sparse import csr_matrix
from scipy.sparse import diags

import importlib.util
from pathlib import Path
import modularity_vitality as mv_v2

<div style="font-size:14pt; font-weight:bold;">Importing Agent x Agent - All Communication</div>

In [2]:
agent_agent_all_comm = ig.read("/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/agent_agent_all_communication.graphml")

<div style="font-size:14pt; font-weight:bold;">Calculating Modularity Vitality Scores</div>

In [ ]:
# Partioning the Network
part = agent_agent_all_comm.community_leiden(objective_function='modularity', weights='weight', resolution_parameter=1)

/tmp/ipykernel_13064/3544820846.py:2: DeprecationWarning: resolution_parameter keyword argument is deprecated, use resolution=... instead
  part = agent_agent_all_comm.community_leiden(objective_function='modularity', weights='weight', resolution_parameter=1)


In [25]:
# Calculating Modularity Vitality Scores
vitalities = mv_v2.modularity_vitality(agent_agent_all_comm, part)

# Map to node identifiers
if "label" in agent_agent_all_comm.vs.attributes():
    names = agent_agent_all_comm.vs["label"]
elif "name" in agent_agent_all_comm.vs.attributes():
    names = agent_agent_all_comm.vs["name"]
else:
    names = list(range(agent_agent_all_comm.vcount()))

In [37]:
# Saving the Partition for Testing
partition_df = pd.DataFrame({
    "Node ID": pd.Series(names, dtype=str),
    "leiden_community": pd.Series(part.membership, dtype=int)
})

partition_path = "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_partition_leiden_gamma1.csv"
os.makedirs(os.path.dirname(partition_path), exist_ok=True)
partition_df.to_csv(partition_path, index=False)
print(f"Saved partition → {partition_path}")

Saved partition → /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_partition_leiden_gamma1.csv


In [29]:
# Partition modularity at γ = 1
Q0 = float(part.modularity)
assert abs(Q0) > 0, "Partition modularity Q0 is zero; normalization undefined."

# Split into hub/bridge columns and build a tidy table
hub = [max(v, 0.0) for v in vitalities]
bridge = [max(-v, 0.0) for v in vitalities]

In [32]:
# Ensure arrays
vitalities = np.asarray(vitalities, dtype=float)
hub = np.maximum(vitalities, 0.0)
bridge = np.maximum(-vitalities, 0.0)

# Normalized columns (ORA-like)
hub_norm = np.where(hub > 0, hub / abs(Q0), 0.0)
bridge_norm = np.where(bridge > 0, bridge / abs(Q0), 0.0)

# (optional) denoise near-zero float chatter
eps = 1e-15
hub_norm = np.where(hub_norm < eps, 0.0, hub_norm)
bridge_norm = np.where(bridge_norm < eps, 0.0, bridge_norm)

# Sanity check lengths
n = agent_agent_all_comm.vcount()
assert len(vitalities) == n and len(names) == n and len(hub) == n and len(bridge) == n

In [35]:
# Construct Table
df = pd.DataFrame({
    "node": names,
    "vitality": vitalities,
    "raw_hub": hub,
    "raw_bridge": bridge,
    "hub_norm": hub_norm,
    "bridge_norm": bridge_norm,
    "community": part.membership  # 0-based community labels from igraph
})

df.head()

,node,vitality,raw_hub,raw_bridge,hub_norm,bridge_norm,community
0,828033366712688640,0.002953,0.002953,0.000000,0.005279,0.000000,0
1,1124877633206931456,-0.002732,0.000000,0.002732,0.000000,0.004885,0
2,8775672,0.003905,0.003905,0.000000,0.006982,0.000000,1
3,1448413205538148357,-0.002028,0.000000,0.002028,0.000000,0.003625,0
4,1163719671117303808,0.002903,0.002903,0.000000,0.005190,0.000000,2


In [38]:
results_path = "/mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_vitality_results_python_gamma1.csv"
df.to_csv(results_path, index=False)
print(f"Saved vitality results → {results_path}")

Saved vitality results → /mnt/d/Dropbox/Netanomics_Resources/Documents/SBP_BRIMS_2025/Large_Graph_Similarity/Test_Data/balikatan_all_comm_vitality_results_python_gamma1.csv


In [ ]:
# Quick Look: Hubs
top_hubs = df.sort_values("hub", ascending=False).head(25)

In [ ]:
# Quick Look: Hubs
top_bridges = df.sort_values("bridge", ascending=False).head(25)